# 08 — Évaluation finale sur le TEST (une seule fois)

**Règle :** le test n'a jamais servi à faire un choix. Tout a été décidé **avant**, sur la validation :

- **Modèle final : SciBERT entraîné sur train + train_plus** (`v2_07_scibert_plus_donnees`), meilleur micro-F1 validation (0,9035).
- **Seuil : 0,5 pour tous les groupes.** Sur la validation, les seuils par groupe faisaient *baisser* le micro-F1
  quand on les testait honnêtement (choisis sur une moitié, mesurés sur l'autre) : 0,9006 contre 0,9035.
- Les autres modèles sont évalués **pour comparaison**, eux aussi au seuil 0,5.

Avant ce notebook, on a vérifié sur la validation que les modèles rechargés ici redonnent exactement les scores de Kaggle.

In [1]:
import sys, re, json
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

sys.path.append("../src")
from metrics import GROUPES, evaluer, rapport_par_groupe, probas_vers_predictions
pd.set_option("display.width", 200)

train =pd.read_csv("../data/processed/train.csv", dtype={"id": str})
train_plus = pd.read_csv("../data/processed/train_plus.csv", dtype={"id": str})
val = pd.read_csv("../data/processed/val.csv", dtype={"id": str})
test = pd.read_csv("../data/processed/test.csv", dtype={"id": str})   # utilisé UNE seule fois
y_val, y_test = val[GROUPES].values, test[GROUPES].values

print("Train :", len(train), "| train_plus :", len(train_plus), "| Validation :", len(val), "| Test :", len(test))

Train : 13993 | train_plus : 19673 | Validation : 3006 | Test : 2983


## 1. Baseline TF-IDF + régression logistique

Mêmes réglages que `03_baseline`. Elle est réentraînée ici (c'est rapide et le résultat est toujours le même) :
une fois sur le train seul, une fois sur train + train_plus.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

def tfidf(df_train):
    vectoriseur = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2),
                                  min_df=2, max_features=100_000, sublinear_tf=True)
    X = vectoriseur.fit_transform(df_train["text"])
    clf = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=4.0), n_jobs=-1)
    clf.fit(X, df_train[GROUPES].values)
    return (clf.predict_proba(vectoriseur.transform(val["text"])),
            clf.predict_proba(vectoriseur.transform(test["text"])))

probas = {}   # nom du modèle -> (probas validation, probas test)
probas["TF-IDF"] = tfidf(train)
probas["TF-IDF (train + plus)"] = tfidf(pd.concat([train, train_plus]))
print("TF-IDF : OK")

TF-IDF : OK


## 2. BiLSTM + GloVe et BiLSTM + attention

Entraînés sur Kaggle (`v2_04b_bilstm`), rechargés ici depuis `models/`. Mêmes classes que dans le notebook Kaggle.

In [3]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

MAX_LEN = 300
with open("../models/bilstm_vocab.json", encoding="utf-8") as f:
    vocab = json.load(f)

def tokeniser(texte):
    return re.findall(r"[a-z0-9]+", texte.lower())

def encoder(df):
    ids, longueurs = [], []
    for texte in df["text"]:
        numeros = [vocab.get(mot, 1) for mot in tokeniser(texte)][:MAX_LEN]
        longueurs.append(len(numeros))
        ids.append(numeros + [0] * (MAX_LEN - len(numeros)))
    return torch.tensor(ids), torch.tensor(longueurs)

class BiLSTM(nn.Module):
    def __init__(self, taille_vocab, dim_emb=200, dim_cachee=128, nb_classes=8, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(taille_vocab, dim_emb, padding_idx=0)
        self.lstm = nn.LSTM(dim_emb, dim_cachee, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.sortie = nn.Linear(2 * dim_cachee, nb_classes)

    def forward(self, ids, longueurs):
        x = self.dropout(self.embedding(ids))
        x = pack_padded_sequence(x, longueurs.cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.lstm(x)
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=ids.size(1))
        h = h.masked_fill(~(ids != 0).unsqueeze(-1), -1e9)
        return self.sortie(self.dropout(h.max(dim=1).values))

class BiLSTMAttention(nn.Module):
    def __init__(self, taille_vocab, dim_emb=200, dim_cachee=128, nb_classes=8, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(taille_vocab, dim_emb, padding_idx=0)
        self.lstm = nn.LSTM(dim_emb, dim_cachee, batch_first=True, bidirectional=True)
        self.attention_w = nn.Linear(2 * dim_cachee, 2 * dim_cachee)
        self.attention_v = nn.Linear(2 * dim_cachee, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.sortie = nn.Linear(2 * dim_cachee, nb_classes)

    def forward(self, ids, longueurs):
        x = self.dropout(self.embedding(ids))
        x = pack_padded_sequence(x, longueurs.cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.lstm(x)
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=ids.size(1))
        scores = self.attention_v(torch.tanh(self.attention_w(h))).squeeze(-1).masked_fill(ids == 0, -1e9)
        poids = torch.softmax(scores, dim=1)
        return self.sortie(self.dropout((poids.unsqueeze(-1) * h).sum(dim=1)))

def probas_lstm(classe, fichier, df):
    modele = classe(len(vocab))
    modele.load_state_dict(torch.load(f"../models/{fichier}", map_location="cpu"))
    modele.eval()   # désactive le dropout
    ids, longueurs = encoder(df)
    with torch.no_grad():
        return torch.cat([torch.sigmoid(modele(ids[i:i + 256], longueurs[i:i + 256]))
                          for i in range(0, len(ids), 256)]).numpy()

probas["BiLSTM + GloVe"] = (probas_lstm(BiLSTM, "bilstm.pt", val), probas_lstm(BiLSTM, "bilstm.pt", test))
probas["BiLSTM + attention + GloVe"] = (probas_lstm(BiLSTMAttention, "attention.pt", val),
                                        probas_lstm(BiLSTMAttention, "attention.pt", test))
print("BiLSTM : OK")

BiLSTM : OK


## 3. SciBERT

Les probabilités ont été calculées sur Kaggle (`v2_05_scibert` et `v2_07_scibert_plus_donnees`).
On vérifie que les articles sont dans le même ordre que `test.csv`.

In [4]:
for nom, fichier in [("SciBERT", "scibert"), ("SciBERT (train + plus)", "scibert_plus_donnees")]:
    ids = pd.read_csv(f"../results/{fichier}_test_ids.csv", dtype={"id": str})["id"]
    assert ids.equals(test["id"]), "ordre des articles différent !"
    probas[nom] = (np.load(f"../results/{fichier}_val_probas.npy"), np.load(f"../results/{fichier}_test_probas.npy"))
print("SciBERT : OK, ordre des articles vérifié")

SciBERT : OK, ordre des articles vérifié


## 4. Tableau final (seuil 0,5)

In [5]:
DONNEES = {"TF-IDF": "train", "TF-IDF (train + plus)": "train + plus", "BiLSTM + GloVe": "train",
           "BiLSTM + attention + GloVe": "train", "SciBERT": "train", "SciBERT (train + plus)": "train + plus"}
lignes = []
for nom, (p_val, p_test) in probas.items():
    s_val = evaluer(y_val, probas_vers_predictions(p_val))
    s_test = evaluer(y_test, probas_vers_predictions(p_test))
    lignes.append({"modèle": nom, "entraînement": DONNEES[nom],
                   "micro-F1 val": s_val["micro_f1"], "micro-F1 test": s_test["micro_f1"],
                   "macro-F1 val": s_val["macro_f1"], "macro-F1 test": s_test["macro_f1"],
                   "subset acc. test": s_test["subset_accuracy"], "hamming test": s_test["hamming_loss"]})
tableau = pd.DataFrame(lignes).set_index("modèle")
tableau.round(4)

,entraînement,micro-F1 val,micro-F1 test,macro-F1 val,macro-F1 test,subset acc. test,hamming test
modèle,,,,,,,
TF-IDF,train,0.8581,0.8611,0.5116,0.5053,0.7804,0.0394
TF-IDF (train + plus),train + plus,0.8736,0.8718,0.6053,0.6055,0.7905,0.0368
BiLSTM + GloVe,train,0.8606,0.8596,0.6406,0.6529,0.7627,0.0422
BiLSTM + attention + GloVe,train,0.8596,0.8597,0.6104,0.5985,0.7663,0.0412
SciBERT,train,0.8953,0.8951,0.7645,0.7557,0.8123,0.0315
SciBERT (train + plus),train + plus,0.9035,0.8996,0.7915,0.7661,0.8176,0.0301


## 5. Modèle final : SciBERT (train + plus) — détail sur le test

In [6]:
p_final_test = probas["SciBERT (train + plus)"][1]
pred = probas_vers_predictions(p_final_test)   # seuil 0,5

# Intervalle de confiance à 95 % (bootstrap : on retire 1000 fois des articles du test au hasard)
rng = np.random.default_rng(42)
boot = [f1_score(y_test[i], pred[i], average="micro")
        for i in (rng.integers(0, len(y_test), len(y_test)) for _ in range(1000))]
bas, haut = np.percentile(boot, [2.5, 97.5])
micro_test = f1_score(y_test, pred, average="micro")
print(f"micro-F1 test : {micro_test:.4f}  (intervalle de confiance 95 % : {bas:.3f} - {haut:.3f})")
print()
print(rapport_par_groupe(y_test, pred))

micro-F1 test : 0.8996  (intervalle de confiance 95 % : 0.891 - 0.907)

              precision    recall  f1-score   support

          cs       0.92      0.91      0.91       974
        math       0.91      0.86      0.89       717
     physics       0.97      0.96      0.97      1498
       q-bio       0.71      0.63      0.67        59
       q-fin       0.78      0.71      0.74        55
        stat       0.71      0.63      0.67       147
        eess       0.59      0.58      0.58       126
        econ       0.76      0.65      0.70        52

   micro avg       0.91      0.89      0.90      3628
   macro avg       0.79      0.74      0.77      3628
weighted avg       0.91      0.89      0.90      3628
 samples avg       0.95      0.93      0.93      3628



In [7]:
# F1 par groupe : validation vs test
f1_val = f1_score(y_val, probas_vers_predictions(probas["SciBERT (train + plus)"][0]), average=None)
f1_test = f1_score(y_test, pred, average=None)
par_groupe = pd.DataFrame({"articles (test)": y_test.sum(0), "F1 val": f1_val, "F1 test": f1_test}, index=GROUPES)
par_groupe.round(3)

,articles (test),F1 val,F1 test
cs,974,0.921,0.913
math,717,0.880,0.886
physics,1498,0.963,0.966
q-bio,59,0.709,0.667
q-fin,55,0.752,0.743
stat,147,0.652,0.669
eess,126,0.662,0.584
econ,52,0.792,0.701


## 6. Analyse des erreurs du modèle final

In [8]:
erreurs = pd.DataFrame({
    "articles du groupe": y_test.sum(0),
    "oubliés (faux négatifs)": ((y_test == 1) & (pred == 0)).sum(0),
    "en trop (faux positifs)": ((y_test == 0) & (pred == 1)).sum(0),
}, index=GROUPES)
print(erreurs)
print()
print(f"Articles parfaitement classés : {100 * (pred == y_test).all(1).mean():.1f} %")
print("Articles sans aucun groupe prédit :", int((pred.sum(1) == 0).sum()))

# Confusions : un groupe oublié et un autre prédit à la place, dans le même article
confusions = Counter()
for vrai, predit in zip(y_test, pred):
    oublies = [g for g, v, p in zip(GROUPES, vrai, predit) if v == 1 and p == 0]
    en_trop = [g for g, v, p in zip(GROUPES, vrai, predit) if v == 0 and p == 1]
    confusions.update((o, e) for o in oublies for e in en_trop)
print("\nConfusions les plus fréquentes (vrai groupe -> prédit à la place) :")
for (o, e), n in confusions.most_common(6):
    print(f"  {o:8s} -> {e:8s} : {n}")

         articles du groupe  oubliés (faux négatifs)  en trop (faux positifs)
cs                      974                       91                       77
math                    717                      100                       59
physics                1498                       57                       45
q-bio                    59                       22                       15
q-fin                    55                       16                       11
stat                    147                       54                       38
eess                    126                       53                       51
econ                     52                       18                       11

Articles parfaitement classés : 81.8 %
Articles sans aucun groupe prédit : 0

Confusions les plus fréquentes (vrai groupe -> prédit à la place) :
  cs       -> physics  : 8
  physics  -> math     : 6
  stat     -> eess     : 6
  math     -> cs       : 6
  math     -> eess     : 5
  physics  -> cs

In [9]:
# Quelques exemples d'erreurs (titre, vrais groupes, groupes prédits)
faux = np.flatnonzero((pred != y_test).any(1))
for i in faux[:8]:
    vrais = [g for g, v in zip(GROUPES, y_test[i]) if v]
    predits = [g for g, p in zip(GROUPES, pred[i]) if p]
    print(f"- {test['title'].iloc[i][:90]}\n    vrais : {vrais} | prédits : {predits}")

- Distributed Consensus Algorithms in Sensor Networks: Quantized Data and Random Link Failur
    vrais : ['cs', 'math'] | prédits : ['cs', 'math', 'eess']
- Modeling Evolving Innovation Networks
    vrais : ['physics'] | prédits : ['cs', 'physics']
- Switching to nonhyperbolic cycles from codim 2 bifurcations of equilibria in ODEs
    vrais : ['math'] | prédits : ['math', 'physics']
- State Estimation Over Wireless Channels Using Multiple Sensors: Asymptotic Behaviour and O
    vrais : ['cs', 'math'] | prédits : ['cs', 'math', 'eess']
- Stochastic Maximum Principle for a PDEs with noise and control on the boundary
    vrais : ['cs', 'math', 'eess'] | prédits : ['math']
- Joint Modelling of Gas and Electricity spot prices
    vrais : ['math', 'q-fin'] | prédits : ['q-fin']
- BRAINSTORMING: Consensus Learning in Practice
    vrais : ['stat'] | prédits : ['cs', 'q-bio', 'stat']
- Inner products of resonance solutions in 1-D quantum barriers
    vrais : ['math', 'physics'] | prédits : ['ph

## 7. Sauvegarde

In [10]:
tableau.round(4).to_csv("../results/evaluation_finale_test.csv")
par_groupe.round(4).to_csv("../results/scibert_final_par_groupe_test.csv")
with open("../results/evaluation_finale_test.json", "w") as f:
    json.dump({"modele_final": "SciBERT (train + plus)", "seuil": 0.5,
               "micro_f1_test": micro_test, "ic95_micro_f1_test": [bas, haut],
               "tous_les_modeles": tableau.round(4).reset_index().to_dict(orient="records")}, f, indent=2, ensure_ascii=False)
print("Sauvegardé dans results/ : evaluation_finale_test.csv, scibert_final_par_groupe_test.csv, evaluation_finale_test.json")

Sauvegardé dans results/ : evaluation_finale_test.csv, scibert_final_par_groupe_test.csv, evaluation_finale_test.json


## Conclusion (écrite après l'unique passage sur le test)

| Modèle | Entraînement | micro-F1 val | **micro-F1 test** | macro-F1 test |
|---|---|---|---|---|
| TF-IDF + régression logistique | train | 0,858 | 0,861 | 0,505 |
| TF-IDF + régression logistique | train + plus | 0,874 | 0,872 | 0,606 |
| BiLSTM + GloVe | train | 0,861 | 0,860 | 0,653 |
| BiLSTM + attention + GloVe | train | 0,860 | 0,860 | 0,599 |
| SciBERT | train | 0,895 | 0,895 | 0,756 |
| **SciBERT (modèle final)** | **train + plus** | **0,904** | **0,900** | **0,766** |

- **Modèle final : micro-F1 test = 0,900** (intervalle de confiance 95 % : 0,891 – 0,907), seuil 0,5.
- Le classement validation = classement test : les choix faits sur la validation tiennent.
- L'écart validation → test est faible (-0,4 pt pour le modèle final, dans l'intervalle de confiance).
- **Le levier le plus fort a été la quantité de données** : +0,5 pt sur le test pour SciBERT (0,895 → 0,900) et +1,1 pt pour TF-IDF.
  L'attention n'apporte rien au BiLSTM ; GloVe l'aide (+2 pts sur la validation).
- Points faibles : les petits groupes (eess 0,58, q-bio 0,67, stat 0,67, econ 0,70 de F1),
  avec seulement 52 à 147 articles dans le test, donc des scores instables. Les erreurs typiques sont des
  catégories croisées (cs/math/physics, stat/eess), souvent ambiguës même pour un humain.